In [4]:
import os
import numpy as np
import pandas as pd
import re

FOLDER = "timing"
CM_REGEX = r"core_10_(\d+)_"

def mad(x):
    """Median Absolute Deviation."""
    med = np.median(x)
    return np.median(np.abs(x - med))

results = []

for filename in os.listdir(FOLDER):
    if not filename.endswith(".txt"):
        continue

    match = re.search(CM_REGEX, filename)
    if not match:
        continue

    cm = int(match.group(1))  # extract CM value (1,2,...)

    filepath = os.path.join(FOLDER, filename)

    # Load file without headers and assign column names manually
    col_names = [
        "energy", "angle", "event_start", "event_end", "len_data",
        "elapsed", "elapsed_dbscan", "elapsed_gmm_sum", "elapsed_merge"
    ]

    df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)

    # Compute stats
    median_ransac = np.median(df["elapsed"])
    mad_ransac    = mad(df["elapsed"])

    median_dbscan = np.median(df["elapsed_dbscan"])
    mad_dbscan    = mad(df["elapsed_dbscan"])

    median_gmm = np.median(df["elapsed_gmm_sum"])
    mad_gmm    = mad(df["elapsed_gmm_sum"])

    median_merge = np.median(df["elapsed_merge"])
    mad_merge    = mad(df["elapsed_merge"])

    median_len_data = np.median(df["len_data"])
    mad_len_data    = mad(df["len_data"])

    results.append({
        "CM": cm,
        "RANSAC Median ± MAD": f"{median_ransac:.4f} ± {mad_ransac:.4f}",
        "DBSCAN Median ± MAD": f"{median_dbscan:.4f} ± {mad_dbscan:.4f}",
        "GMM Median ± MAD": f"{median_gmm:.4f} ± {mad_gmm:.4f}",
        "MERGE Median ± MAD": f"{median_merge:.4f} ± {mad_merge:.4f}",
        "len_data Median ± MAD": f"{median_len_data:.0f} ± {mad_len_data:.0f}"
    })

# Convert to DataFrame and sort by CM
results_df = pd.DataFrame(results).sort_values("CM")

print("\n=== Timing Summary (Median ± MAD) ===\n")
print(results_df.to_string(index=False))



=== Timing Summary (Median ± MAD) ===

 CM RANSAC Median ± MAD DBSCAN Median ± MAD GMM Median ± MAD MERGE Median ± MAD len_data Median ± MAD
  1     2.9500 ± 0.0377     0.0094 ± 0.0005  1.1340 ± 0.1015    0.0076 ± 0.0023              476 ± 19
  2     2.9918 ± 0.0331     0.0097 ± 0.0006  1.1275 ± 0.1183    0.0078 ± 0.0030              504 ± 20
  3     2.8342 ± 0.0407     0.0093 ± 0.0006  1.1742 ± 0.1934    0.0112 ± 0.0052              529 ± 31
  4     3.0044 ± 0.0934     0.0100 ± 0.0007  1.2753 ± 0.2499    0.0122 ± 0.0049              559 ± 44
  5     2.9631 ± 0.2086     0.0101 ± 0.0007  1.4066 ± 0.3050    0.0147 ± 0.0074              594 ± 62


C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\2634120189.py:34: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\2634120189.py:34: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\2634120189.py:34: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\2634120189.py:34: FutureWarning: The 'delim_whitespace' keywor

In [5]:
import os
import numpy as np
import pandas as pd
import re

# Folders and their RANSAC iteration labels
FOLDERS = {
    "timing": "5000",      # previous folder, iterations=5000
    "timing_1000": "1000"  # new folder, iterations=1000
}

CM_REGEX = r"core_10_(\d+)_"

def mad(x):
    """Median Absolute Deviation."""
    med = np.median(x)
    return np.median(np.abs(x - med))

# Initialize results dictionary
results_dict = {}

for folder, iter_label in FOLDERS.items():
    for filename in os.listdir(folder):
        if not filename.endswith(".txt"):
            continue

        match = re.search(CM_REGEX, filename)
        if not match:
            continue

        cm = int(match.group(1))  # extract CM value

        filepath = os.path.join(folder, filename)

        # Load file without headers and assign column names manually
        col_names = [
            "energy", "angle", "event_start", "event_end", "len_data",
            "elapsed", "elapsed_dbscan", "elapsed_gmm_sum", "elapsed_merge"
        ]
        df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)

        # Compute statistics
        median_ransac = np.median(df["elapsed"])
        mad_ransac    = mad(df["elapsed"])

        median_dbscan = np.median(df["elapsed_dbscan"])
        mad_dbscan    = mad(df["elapsed_dbscan"])

        median_gmm = np.median(df["elapsed_gmm_sum"])
        mad_gmm    = mad(df["elapsed_gmm_sum"])

        median_merge = np.median(df["elapsed_merge"])
        mad_merge    = mad(df["elapsed_merge"])

        median_len_data = np.median(df["len_data"])
        mad_len_data    = mad(df["len_data"])

        # Initialize entry if not exist
        if cm not in results_dict:
            results_dict[cm] = {
                "DBSCAN Median ± MAD": f"{median_dbscan:.4f} ± {mad_dbscan:.4f}",
                "GMM Median ± MAD": f"{median_gmm:.4f} ± {mad_gmm:.4f}",
                "MERGE Median ± MAD": f"{median_merge:.4f} ± {mad_merge:.4f}",
                "len_data Median ± MAD": f"{median_len_data:.0f} ± {mad_len_data:.0f}"
            }

        # Add RANSAC timing for this iteration count
        results_dict[cm][f"RANSAC_{iter_label} Median ± MAD"] = f"{median_ransac:.4f} ± {mad_ransac:.4f}"

# Convert results to DataFrame and sort by CM
results_df = pd.DataFrame([
    {"CM": cm, **metrics} for cm, metrics in results_dict.items()
]).sort_values("CM")

print("\n=== Timing Summary (Median ± MAD) ===\n")
print(results_df.to_string(index=False))



=== Timing Summary (Median ± MAD) ===

 CM DBSCAN Median ± MAD GMM Median ± MAD MERGE Median ± MAD len_data Median ± MAD RANSAC_5000 Median ± MAD RANSAC_1000 Median ± MAD
  1     0.0094 ± 0.0005  1.1340 ± 0.1015    0.0076 ± 0.0023              476 ± 19          2.9500 ± 0.0377          0.5988 ± 0.0085
  2     0.0097 ± 0.0006  1.1275 ± 0.1183    0.0078 ± 0.0030              504 ± 20          2.9918 ± 0.0331          0.5985 ± 0.0083
  3     0.0093 ± 0.0006  1.1742 ± 0.1934    0.0112 ± 0.0052              529 ± 31          2.8342 ± 0.0407          0.5725 ± 0.0126
  4     0.0100 ± 0.0007  1.2753 ± 0.2499    0.0122 ± 0.0049              559 ± 44          3.0044 ± 0.0934          0.6099 ± 0.0243
  5     0.0101 ± 0.0007  1.4066 ± 0.3050    0.0147 ± 0.0074              594 ± 62          2.9631 ± 0.2086          0.6249 ± 0.0919


C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\762016704.py:40: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\762016704.py:40: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\762016704.py:40: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\762016704.py:40: FutureWarning: The 'delim_whitespace' keyword in

In [6]:
import os
import numpy as np
import pandas as pd
import re

# Folders and their RANSAC iteration labels
FOLDERS = {
    "timing": "5000",      # previous folder, iterations=5000
    "timing_1000": "1000"  # new folder, iterations=1000
}

CM_REGEX = r"core_10_(\d+)_"

def mad(x):
    """Median Absolute Deviation."""
    med = np.median(x)
    return np.median(np.abs(x - med))

results_dict = {}

for folder, iter_label in FOLDERS.items():
    for filename in os.listdir(folder):
        if not filename.endswith(".txt"):
            continue

        match = re.search(CM_REGEX, filename)
        if not match:
            continue

        cm = int(match.group(1))  # extract CM value

        filepath = os.path.join(folder, filename)

        # Load file without headers and assign column names manually
        col_names = [
            "energy", "angle", "event_start", "event_end", "len_data",
            "elapsed", "elapsed_dbscan", "elapsed_gmm_sum", "elapsed_merge"
        ]
        df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)

        # Compute stats
        median_ransac = np.median(df["elapsed"])
        mad_ransac    = mad(df["elapsed"])

        median_dbscan = np.median(df["elapsed_dbscan"])
        mad_dbscan    = mad(df["elapsed_dbscan"])

        median_gmm = np.median(df["elapsed_gmm_sum"])
        mad_gmm    = mad(df["elapsed_gmm_sum"])

        median_merge = np.median(df["elapsed_merge"])
        mad_merge    = mad(df["elapsed_merge"])

        median_len_data = np.median(df["len_data"])
        mad_len_data    = mad(df["len_data"])

        # Initialize entry if not exist
        if cm not in results_dict:
            results_dict[cm] = {
                "len_data Median ± MAD": f"{median_len_data:.0f} ± {mad_len_data:.0f}"
            }

        # Convert DBSCAN and MERGE to milliseconds and round to 2 decimals
        results_dict[cm]["DBSCAN Median ± MAD (ms)"] = f"{median_dbscan*1000:.2f} ± {mad_dbscan*1000:.2f}"
        results_dict[cm]["MERGE Median ± MAD (ms)"] = f"{median_merge*1000:.2f} ± {mad_merge*1000:.2f}"

        # Keep RANSAC and GMM in seconds, round to 2 decimals
        results_dict[cm][f"RANSAC_{iter_label} Median ± MAD (s)"] = f"{median_ransac:.2f} ± {mad_ransac:.2f}"
        results_dict[cm]["GMM Median ± MAD (s)"] = f"{median_gmm:.2f} ± {mad_gmm:.2f}"

# Convert results to DataFrame and sort by CM
results_df = pd.DataFrame([
    {"CM": cm, **metrics} for cm, metrics in results_dict.items()
]).sort_values("CM")

print("\n=== Timing Summary ===\n")
print(results_df.to_string(index=False))



=== Timing Summary ===

 CM len_data Median ± MAD DBSCAN Median ± MAD (ms) MERGE Median ± MAD (ms) RANSAC_5000 Median ± MAD (s) GMM Median ± MAD (s) RANSAC_1000 Median ± MAD (s)
  1              476 ± 19              8.67 ± 0.66             7.78 ± 2.43                  2.95 ± 0.04          1.17 ± 0.10                  0.60 ± 0.01
  2              504 ± 20              9.24 ± 0.64             7.95 ± 3.07                  2.99 ± 0.03          1.15 ± 0.13                  0.60 ± 0.01
  3              529 ± 31              8.72 ± 0.55            11.68 ± 5.67                  2.83 ± 0.04          1.19 ± 0.20                  0.57 ± 0.01
  4              559 ± 44             10.14 ± 0.69            12.96 ± 5.40                  3.00 ± 0.09          1.30 ± 0.25                  0.61 ± 0.02
  5              594 ± 62             10.51 ± 0.77            15.14 ± 7.44                  2.96 ± 0.21          1.48 ± 0.31                  0.62 ± 0.09


C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\979411308.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\979411308.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\979411308.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath, delim_whitespace=True, names=col_names, header=None)
C:\Users\alarokia\AppData\Local\Temp\ipykernel_18144\979411308.py:39: FutureWarning: The 'delim_whitespace' keyword in